# Notebook 02 - NLP: Text Processing

Goes with **Lesson 02**.

## 0. Download the NLTK language files (only the first time)

NLTK needs some small files (word lists, dictionaries). This needs internet once and takes about a minute.

In [ ]:
import nltk

needed = ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
          "averaged_perceptron_tagger_eng", "maxent_ne_chunker_tab", "words", "vader_lexicon"]
for name in needed:
    nltk.download(name, quiet=True)
print("NLTK data ready.")

In [ ]:
from pathlib import Path

# The data folder is next to the notebooks folder. In Colab, upload the files to the current folder.
DATA_DIR = Path("../data") if Path("../data").exists() else Path(".")
DOCS_DIR = DATA_DIR / "documents" if (DATA_DIR / "documents").exists() else DATA_DIR
print("Data folder:", DATA_DIR.resolve())

## 1. Cleaning and tokenisation

In [ ]:
from nltk.tokenize import word_tokenize, sent_tokenize

text = "The students were studying hard for their exams. They studied better after sleeping well! Studies show that sleep helps memory."

print("Sentences:")
for s in sent_tokenize(text):
    print(" -", s)

tokens = word_tokenize(text.lower())
print()
print("Word tokens:", tokens)

## 2. Remove stop words (and punctuation)

In [ ]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))
print("Some English stop words:", sorted(stop_words)[:15], "...")

clean_tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
print()
print("After removing stop words and punctuation:", clean_tokens)

## 3. Stemming vs lemmatisation

In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

print(f"{'word':12} {'stem':10} {'lemma (as verb)':15}")
for word in ["studying", "studied", "studies", "sleeping", "better", "running", "ran"]:
    print(f"{word:12} {stemmer.stem(word):10} {lemmatizer.lemmatize(word, pos='v'):15}")

print()
print("'better' as an adjective ->", lemmatizer.lemmatize("better", pos="a"))

Stems can be non-words (`studi`). Lemmas are real dictionary words (`study`).

## 4. N-grams

In [ ]:
from nltk.util import ngrams

sentence = "i do not like this phone".split()
print("Unigrams:", list(ngrams(sentence, 1)))
print("Bigrams: ", list(ngrams(sentence, 2)))
print("Trigrams:", list(ngrams(sentence, 3)))

## 5. Bag of Words: count the words

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

small_texts = ["very good movie", "bad movie", "very very bad acting"]
bow = CountVectorizer()
counts = bow.fit_transform(small_texts)

pd.DataFrame(counts.toarray(), columns=bow.get_feature_names_out(), index=small_texts)

## 6. TF-IDF: important AND rare words get high scores

We use the 3 documents in `data/documents/`.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

doc_files = sorted(DOCS_DIR.glob("*.txt"))
doc_names = [f.stem for f in doc_files]
doc_texts = [f.read_text(encoding="utf-8") for f in doc_files]
print("Documents:", doc_names)

tfidf = TfidfVectorizer(stop_words="english")
scores = tfidf.fit_transform(doc_texts)
table = pd.DataFrame(scores.toarray(), columns=tfidf.get_feature_names_out(), index=doc_names)

print()
print("Top 5 words in each document by TF-IDF:")
for name in doc_names:
    print(f"  {name}: {list(table.loc[name].sort_values(ascending=False).head(5).index)}")

print()
table[["jupiter", "library", "sleep", "students"]].round(3)

`jupiter` scores only in the solar system notes. `students` appears in two documents, so it is less special.

## 7. Word embeddings: the idea

Real embeddings have hundreds of numbers learned from huge text. Here we make tiny **hand-made** vectors with 4 meanings: `[royal, male, female, fruit]`, just to see the famous idea.

In [ ]:
import numpy as np

word_vectors = {
    "king":  np.array([0.9, 0.9, 0.1, 0.0]),
    "queen": np.array([0.9, 0.1, 0.9, 0.0]),
    "man":   np.array([0.1, 0.9, 0.1, 0.0]),
    "woman": np.array([0.1, 0.1, 0.9, 0.0]),
    "apple": np.array([0.0, 0.0, 0.0, 1.0]),
}

def cosine(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

result = word_vectors["king"] - word_vectors["man"] + word_vectors["woman"]
print("king - man + woman =", result.round(2))
for word, vector in word_vectors.items():
    print(f"  similarity to {word:6}: {cosine(result, vector):.2f}")

The closest word is **queen**. Word2Vec learns vectors like these **automatically** by reading millions of sentences. You will use real embeddings in notebook 07.

## 8. Named Entity Recognition (NER)

In [ ]:
from nltk import pos_tag, ne_chunk

sentence = "Sundar Pichai visited Bengaluru on Monday to meet engineers from Infosys and Google."
try:
    tree = ne_chunk(pos_tag(word_tokenize(sentence)))
    for subtree in tree:
        if hasattr(subtree, "label"):
            entity = " ".join(word for word, tag in subtree.leaves())
            print(f"{entity:15} -> {subtree.label()}")
except LookupError as error:
    print("NER files are missing. Run the download cell again. (This step is optional.)")

NLTK's NER is old and simple, so it makes mistakes (for example it may label a city as a PERSON, or split one name into two). Modern tools like **spaCy** or LLMs do much better.

## 9. Sentiment analysis

### 9a. Rule-based (lexicon): VADER

`compound` goes from -1 (very negative) to +1 (very positive).

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer

vader = SentimentIntensityAnalyzer()
for review in ["I love this phone, the camera is amazing!",
               "The delivery was late and the box was damaged.",
               "It is okay.",
               "Not bad at all"]:
    print(f"{vader.polarity_scores(review)['compound']:+.2f}  {review}")

### 9b. Machine learning: train our own text classifier on `reviews.csv`

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

reviews = pd.read_csv(DATA_DIR / "reviews.csv")
print(reviews["sentiment"].value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    reviews["review"], reviews["sentiment"], test_size=0.25, random_state=42, stratify=reviews["sentiment"])

classifier = make_pipeline(TfidfVectorizer(ngram_range=(1, 1)), LogisticRegression())
classifier.fit(X_train, y_train)
print()
print(f"Test accuracy: {classifier.score(X_test, y_test):.2f}  (only 45 training reviews, so this is a small demo)")

In [ ]:
my_reviews = [
    "The camera is excellent and the battery is great",
    "Terrible quality, it broke and I want a refund",
    "Fast delivery, works perfectly",
    "The keyboard is noisy and the screen cracked",
]
for review, label in zip(my_reviews, classifier.predict(my_reviews)):
    print(f"{label:9} <- {review}")

**Try it:** add your own reviews to `my_reviews`. Then change `ngram_range=(1, 1)` to `(1, 2)` and test "not good".

## 10. Extractive summarisation

Score each sentence by how many **frequent important words** it contains, then keep the top sentences in their original order.

In [ ]:
from collections import Counter

def summarise(text, n_sentences=3):
    # keep only real sentences (headings have no full stop at the end)
    text = " ".join(line.strip() for line in text.splitlines() if line.strip().endswith("."))
    sentences = sent_tokenize(text)
    words = [w for w in word_tokenize(text.lower()) if w.isalpha() and w not in stop_words]
    frequency = Counter(words)
    scores = {}
    for i, sentence in enumerate(sentences):
        sentence_words = [w for w in word_tokenize(sentence.lower()) if w.isalpha()]
        scores[i] = sum(frequency.get(w, 0) for w in sentence_words) / (len(sentence_words) + 1)
    best = sorted(sorted(scores, key=scores.get, reverse=True)[:n_sentences])
    return " ".join(sentences[i] for i in best)

notes = (DOCS_DIR / "solar_system_notes.txt").read_text(encoding="utf-8")
print("Original length:", len(notes.split()), "words")
print()
print(summarise(notes, 3))

This is **extractive**: every sentence is copied from the original. In notebook 06 an LLM writes an **abstractive** summary in its own words.

Go back to **Lesson 02** and do the **Check** section.